In [8]:
import os
import matplotlib
# 强制使用 Agg 后端，必须在 import pyplot 之前调用
matplotlib.use('Agg') 

import numpy as np
import matplotlib.pyplot as plt
# ... 其余导入保持不变
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.stats import gaussian_kde
from matplotlib import rcParams
import os

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['font.size'] = 14
rcParams['figure.dpi'] = 600

def draw_figB():
    if not os.path.exists("data_BC.npz"):
        print("错误: 找不到 data_BC.npz")
        return

    data = np.load("data_BC.npz")
    features = data['features']
    true_labels = data['labels']

    raw_dists = pdist(features, metric='euclidean')

    scaled_dists = raw_dists
    dist_matrix = squareform(scaled_dists)

    iu1 = np.triu_indices(100, 1)
    same_tiger = (true_labels[:, None] == true_labels[None, :])

    intra_dists = dist_matrix[same_tiger][dist_matrix[same_tiger] > 0] 
    inter_dists = scaled_dists[~same_tiger[iu1]] 

    figB, axB = plt.subplots(figsize=(7, 5))
    sns.kdeplot(intra_dists, fill=True, color="#3C5488", label="Intra-class Distance", alpha=0.6, linewidth=2, ax=axB)
    sns.kdeplot(inter_dists, fill=True, color="#DC0000", label="Inter-class Distance", alpha=0.6, linewidth=2, ax=axB)

    kde_intra = gaussian_kde(intra_dists)
    kde_inter = gaussian_kde(inter_dists)
    x_eval = np.linspace(0, 150, 1000)
    diff = kde_intra(x_eval) - kde_inter(x_eval)
    crossings = np.where(np.diff(np.sign(diff)))[0]
    
    valid_crossings = [x_eval[i] for i in crossings if intra_dists.mean() < x_eval[i] < inter_dists.mean()]
    decision_boundary = valid_crossings[0] if valid_crossings else (intra_dists.mean() + inter_dists.mean())/2

    axB.axvline(x=decision_boundary, color='#333333', linestyle='--', linewidth=2, zorder=4)
    axB.text(decision_boundary + 2, axB.get_ylim()[1]*0.8, 'Decision\nBoundary', 
             ha='left', va='bottom', fontweight='bold', fontsize=12)

    axB.set_xlabel("Euclidean Distance", fontweight='bold')
    axB.set_ylabel("Density", fontweight='bold')
    axB.spines['top'].set_visible(False)
    axB.spines['right'].set_visible(False)
    axB.legend(frameon=False, loc='upper right', fontsize=12)

    plt.tight_layout()
    plt.savefig("Fig2b_Distance_Distribution.png")
    print(f"✅ Fig 2b saved！决策红线={decision_boundary:.1f}")

if __name__ == "__main__":
    draw_figB()

✅ Fig 2b saved！决策红线=59.3


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from matplotlib import rcParams
import os

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['font.size'] = 14
rcParams['figure.dpi'] = 600

def draw_figA():
    if not os.path.exists("data_A.npz"):
        print("错误: 找不到 data_A.npz")
        return

    data = np.load("data_A.npz")
    features = data['features']
    labels_A = data['labels']

    raw_dists = pdist(features, metric='euclidean')
    scaled_dists = raw_dists
    dist_matrix = squareform(scaled_dists)

    figA, axA = plt.subplots(figsize=(6, 5))
    sns.heatmap(dist_matrix, annot=True, fmt=".1f", cmap="YlGnBu", 
                xticklabels=labels_A, yticklabels=labels_A, 
                cbar_kws={'label': 'Euclidean Distance'}, 
                ax=axA, annot_kws={"weight": "bold", "size": 13})
                

    
    plt.tight_layout()
    plt.savefig("Fig2a_Distance_Matrix.png")
    print("✅ Fig 2a saved！")

if __name__ == "__main__":
    draw_figA()

✅ Fig 2a saved！
